In [1]:
import torch
import torch.nn as nn
import numpy as np
import os
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report

print("Tudo importado!")

Tudo importado!


In [2]:
class CNN1D_EEG(nn.Module):
    def __init__(self, num_classes, pontos_por_janela=346):
        super().__init__()
        
        self.conv1 = nn.Conv1d(in_channels=1, out_channels=16, kernel_size=21, padding=10)
        self.conv2 = nn.Conv1d(in_channels=16, out_channels=32, kernel_size=11, padding=5)
        self.conv3 = nn.Conv1d(in_channels=32, out_channels=64, kernel_size=7, padding=3)
        self.pool = nn.MaxPool1d(kernel_size=2)
        
        with torch.no_grad():
            ficticio = torch.zeros(1, 1, pontos_por_janela)
            ficticio = self.pool(self.conv1(ficticio))
            ficticio = self.pool(self.conv2(ficticio))
            ficticio = self.pool(self.conv3(ficticio))
            tamanho_achatado = ficticio.view(1, -1).shape[1]
        
        self.fc1 = nn.Linear(tamanho_achatado, 64)
        self.fc2 = nn.Linear(64, num_classes)
        
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(p=0.2)
    
    def forward(self, x):
        x = self.pool(self.relu(self.conv1(x)))
        x = self.pool(self.relu(self.conv2(x)))
        x = self.pool(self.relu(self.conv3(x)))
        x = x.view(x.size(0), -1)
        x = self.dropout(self.relu(self.fc1(x)))
        x = self.fc2(x)
        return x

print("Classe definida!")

Classe definida!


In [3]:
# Modelo raiz: 3 classes (Saudável, Interictal, Crise)
modelo_ternario = CNN1D_EEG(num_classes=3)
modelo_ternario.load_state_dict(torch.load("modelo_ternario.pth"))
modelo_ternario.eval()

# Modelo folha 1: 2 classes (A, B)
modelo_a_vs_b = CNN1D_EEG(num_classes=2)
modelo_a_vs_b.load_state_dict(torch.load("modelo_a_vs_b.pth"))
modelo_a_vs_b.eval()

# Modelo folha 2: 2 classes (C, D)
modelo_c_vs_d = CNN1D_EEG(num_classes=2)
modelo_c_vs_d.load_state_dict(torch.load("modelo_c_vs_d.pth"))
modelo_c_vs_d.eval()

print("Os 3 modelos foram carregados com sucesso!")

Os 3 modelos foram carregados com sucesso!


In [4]:
caminho_base = "."

sets = {
    "Z": {"nome": "Set A - Olhos abertos", "rotulo": 0},
    "O": {"nome": "Set B - Olhos fechados", "rotulo": 1},
    "N": {"nome": "Set C - Fora da zona", "rotulo": 2},
    "F": {"nome": "Set D - Zona epileptogênica", "rotulo": 3},
    "S": {"nome": "Set E - Crise", "rotulo": 4},
}

sinais = []
rotulos = []

for pasta, info in sets.items():
    caminho_pasta = os.path.join(caminho_base, pasta)
    arquivos = sorted([f for f in os.listdir(caminho_pasta) if f.lower().endswith('.txt')])
    
    for arquivo in arquivos:
        caminho_arquivo = os.path.join(caminho_pasta, arquivo)
        sinal = np.loadtxt(caminho_arquivo)
        sinais.append(sinal)
        rotulos.append(info["rotulo"])

sinais = np.array(sinais, dtype=np.float32)
rotulos = np.array(rotulos, dtype=np.int64)

print(f"Total de sinais: {len(sinais)}")
print(f"Classe 0 (A): {(rotulos == 0).sum()}, Classe 1 (B): {(rotulos == 1).sum()}, "
      f"Classe 2 (C): {(rotulos == 2).sum()}, Classe 3 (D): {(rotulos == 3).sum()}, "
      f"Classe 4 (E): {(rotulos == 4).sum()}")

Total de sinais: 500
Classe 0 (A): 100, Classe 1 (B): 100, Classe 2 (C): 100, Classe 3 (D): 100, Classe 4 (E): 100


In [5]:
duracao_janela = 2.0
taxa = 173.61
pontos_por_janela = int(duracao_janela * taxa)

janelas = []
rotulos_janelas = []

for i in range(len(sinais)):
    sinal = sinais[i]
    rotulo = rotulos[i]
    num_janelas = len(sinal) // pontos_por_janela
    
    for j in range(num_janelas):
        inicio = j * pontos_por_janela
        fim = inicio + pontos_por_janela
        janela = sinal[inicio:fim]
        janelas.append(janela)
        rotulos_janelas.append(rotulo)

janelas = np.array(janelas, dtype=np.float32)
rotulos_janelas = np.array(rotulos_janelas, dtype=np.int64)

print(f"Total de janelas: {len(janelas)}")
print(f"Pontos por janela: {pontos_por_janela}")

Total de janelas: 5500
Pontos por janela: 347


In [6]:
janelas_treino, janelas_teste, rot_treino, rot_teste = train_test_split(
    janelas, rotulos_janelas, test_size=0.2, random_state=42, stratify=rotulos_janelas
)

print(f"Teste: {len(janelas_teste)} janelas")
for classe in range(5):
    print(f"  Classe {classe}: {(rot_teste == classe).sum()} janelas")

Teste: 1100 janelas
  Classe 0: 220 janelas
  Classe 1: 220 janelas
  Classe 2: 220 janelas
  Classe 3: 220 janelas
  Classe 4: 220 janelas


In [7]:
# Normalização para o modelo ternário (usa TODOS os dados de treino: A+B+C+D+E)
media_ternario = janelas_treino.mean()
desvio_ternario = janelas_treino.std()

# Normalização para o modelo A vs B (usa só as janelas de treino das classes 0 e 1)
mascara_ab_treino = (rot_treino == 0) | (rot_treino == 1)
media_ab = janelas_treino[mascara_ab_treino].mean()
desvio_ab = janelas_treino[mascara_ab_treino].std()

# Normalização para o modelo C vs D (usa só as janelas de treino das classes 2 e 3)
mascara_cd_treino = (rot_treino == 2) | (rot_treino == 3)
media_cd = janelas_treino[mascara_cd_treino].mean()
desvio_cd = janelas_treino[mascara_cd_treino].std()

print(f"Normalização ternário: média={media_ternario:.4f}, desvio={desvio_ternario:.4f}")
print(f"Normalização A vs B:   média={media_ab:.4f}, desvio={desvio_ab:.4f}")
print(f"Normalização C vs D:   média={media_cd:.4f}, desvio={desvio_cd:.4f}")

Normalização ternário: média=-7.2811, desvio=164.1872
Normalização A vs B:   média=-8.9434, desvio=60.7234
Normalização C vs D:   média=-7.0958, desvio=75.7055


In [8]:
# Mantém as janelas de teste como array numpy por enquanto
# (vamos normalizar sob demanda, dependendo de qual modelo for usado)
print(f"Formato das janelas de teste: {janelas_teste.shape}")
print(f"Rótulos de teste (5 classes): {np.unique(rot_teste)}")

Formato das janelas de teste: (1100, 347)
Rótulos de teste (5 classes): [0 1 2 3 4]


In [9]:
def prever_cascata(janela_bruta):
    """
    Recebe uma única janela de EEG (não normalizada) e retorna
    a previsão final entre as 5 classes originais (0=A, 1=B, 2=C, 3=D, 4=E).
    """
    
    # --- ETAPA 1: modelo raiz (ternário) ---
    janela_norm_ternario = (janela_bruta - media_ternario) / desvio_ternario
    tensor_ternario = torch.FloatTensor(janela_norm_ternario).unsqueeze(0).unsqueeze(0)
    # unsqueeze(0) duas vezes: uma para "batch" de tamanho 1, outra para "canal"
    
    with torch.no_grad():
        saida_ternario = modelo_ternario(tensor_ternario)
        previsao_raiz = saida_ternario.argmax(dim=1).item()
    # previsao_raiz: 0 = Saudável, 1 = Interictal, 2 = Crise
    
    # --- ETAPA 2: roteamento para o modelo folha adequado ---
    
    if previsao_raiz == 0:
        # Saudável → usa modelo A vs B
        janela_norm_ab = (janela_bruta - media_ab) / desvio_ab
        tensor_ab = torch.FloatTensor(janela_norm_ab).unsqueeze(0).unsqueeze(0)
        
        with torch.no_grad():
            saida_ab = modelo_a_vs_b(tensor_ab)
            previsao_folha = saida_ab.argmax(dim=1).item()
        # previsao_folha: 0 = A, 1 = B → já bate com o esquema de 5 classes
        return previsao_folha
    
    elif previsao_raiz == 1:
        # Interictal → usa modelo C vs D
        janela_norm_cd = (janela_bruta - media_cd) / desvio_cd
        tensor_cd = torch.FloatTensor(janela_norm_cd).unsqueeze(0).unsqueeze(0)
        
        with torch.no_grad():
            saida_cd = modelo_c_vs_d(tensor_cd)
            previsao_folha = saida_cd.argmax(dim=1).item()
        # previsao_folha: 0 = C, 1 = D → precisa remapear para 2 e 3
        return previsao_folha + 2
    
    else:
        # Crise → resposta final direta, sem segundo modelo
        return 4  # classe E

print("Função da cascata definida!")

Função da cascata definida!


In [10]:
previsoes_cascata = []

for janela in janelas_teste:
    previsao = prever_cascata(janela)
    previsoes_cascata.append(previsao)

previsoes_cascata = np.array(previsoes_cascata)

acuracia_cascata = (previsoes_cascata == rot_teste).mean() * 100
print(f"Acurácia da cascata (5 classes): {acuracia_cascata:.2f}%")
print(f"Acertos: {(previsoes_cascata == rot_teste).sum()}/{len(rot_teste)}")

Acurácia da cascata (5 classes): 90.73%
Acertos: 998/1100


In [11]:
nomes_classes = ["A (olhos abertos)", "B (olhos fechados)", "C (fora zona)", "D (zona epilept.)", "E (crise)"]

cm = confusion_matrix(rot_teste, previsoes_cascata)

print("=== MATRIZ DE CONFUSÃO — CASCATA (5 CLASSES) ===\n")
cabecalho = f"{'':>20}" + "".join([f"{n[:8]:>10}" for n in nomes_classes])
print(cabecalho)
for i, nome in enumerate(nomes_classes):
    linha = f"{nome:>20}" + "".join([f"{cm[i][j]:>10}" for j in range(5)])
    print(linha)

print(f"\n=== RELATÓRIO DE CLASSIFICAÇÃO ===\n")
print(classification_report(rot_teste, previsoes_cascata, target_names=nomes_classes))

=== MATRIZ DE CONFUSÃO — CASCATA (5 CLASSES) ===

                      A (olhos  B (olhos  C (fora   D (zona   E (crise
   A (olhos abertos)       179        31         6         4         0
  B (olhos fechados)        33       186         0         1         0
       C (fora zona)         2         1       206        10         1
   D (zona epilept.)         1         0         9       210         0
           E (crise)         0         0         0         3       217

=== RELATÓRIO DE CLASSIFICAÇÃO ===

                    precision    recall  f1-score   support

 A (olhos abertos)       0.83      0.81      0.82       220
B (olhos fechados)       0.85      0.85      0.85       220
     C (fora zona)       0.93      0.94      0.93       220
 D (zona epilept.)       0.92      0.95      0.94       220
         E (crise)       1.00      0.99      0.99       220

          accuracy                           0.91      1100
         macro avg       0.91      0.91      0.91      1100
     

In [12]:
# Mapeamento de classe original para categoria da raiz
def categoria_raiz(classe_5):
    if classe_5 in [0, 1]:
        return 0  # Saudável
    elif classe_5 in [2, 3]:
        return 1  # Interictal
    else:
        return 2  # Crise

categoria_raiz_real = np.array([categoria_raiz(c) for c in rot_teste])
categoria_raiz_prevista = np.array([categoria_raiz(c) for c in previsoes_cascata])

acuracia_raiz = (categoria_raiz_real == categoria_raiz_prevista).mean() * 100
print(f"Acurácia da ETAPA 1 (categoria ampla, dentro da cascata): {acuracia_raiz:.2f}%")

# Dentro dos casos em que a raiz acertou a categoria, quantos a folha também acertou?
mascara_raiz_correta = categoria_raiz_real == categoria_raiz_prevista
acuracia_folha_dado_raiz_correta = (previsoes_cascata[mascara_raiz_correta] == rot_teste[mascara_raiz_correta]).mean() * 100
print(f"Acurácia da folha, DADO que a raiz acertou: {acuracia_folha_dado_raiz_correta:.2f}%")
print(f"(Baseado em {mascara_raiz_correta.sum()} janelas onde a raiz acertou)")

Acurácia da ETAPA 1 (categoria ampla, dentro da cascata): 98.27%
Acurácia da folha, DADO que a raiz acertou: 92.32%
(Baseado em 1081 janelas onde a raiz acertou)
